In [4]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
ratings_df = pd.read_csv('dataset/movies/ratings.csv')
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [2]:
movies_metadata = pd.read_csv("dataset/movies/movies_metadata.csv")
movies_metadata = movies_metadata[['title', 'belongs_to_collection', 'id','genres']]

#rename column 'id' to 'movieId' 
movies_metadata = movies_metadata.rename(columns={"id": "movieId", 'belongs_to_collection': 'collection'})

#Convert data type of 'movieId' from string int64 and making any 'movieId' thats not a number to NaN then dropping them
movies_metadata['movieId'] = pd.to_numeric(movies_metadata['movieId'], errors='coerce')
movies_metadata = movies_metadata.dropna(subset=['movieId'])
movies_metadata['movieId'] = movies_metadata['movieId'].astype('int64')

movies_metadata.head()

/var/folders/7k/s2jvnzk97s987g3tmxp90kdw0000gn/T/ipykernel_59581/2477776612.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv("dataset/movies/movies_metadata.csv")


,title,collection,movieId,genres
0,Toy Story,"{'id': 10194, 'name': 'Toy Story Collection', ...",862,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,NaN,8844,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",15602,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,NaN,31357,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,"{'id': 96871, 'name': 'Father of the Bride Col...",11862,"[{'id': 35, 'name': 'Comedy'}]"


In [ ]:
#merge user_ratings and movie_metadata on 'movieId'
movie_data = ratings_df.merge(movies_metadata, on='movieId')
movie_data = movie_data[['userId', 'movieId', 'rating', 'title', 'genres']]
movie_data.head()

KeyError: "['belongs_to_collection'] not in index"

In [ ]:
#Extract the different genres and put them into an array without their corresponding ids
movies_metadata['genres'] = movies_metadata['genres'].apply(lambda x: [item['name'] for item in ast.literal_eval(x)])

#Extract the collection the movie belongs to if any
movies_metadata['collection'] = movies_metadata['collection'].apply(lambda x: ast.literal_eval(x)['name'] if pd.notna(x) and x != '' else '')

In [8]:
movies_metadata.head()

,title,collection,movieId,genres
0,Toy Story,Toy Story Collection,862,"[Animation, Comedy, Family]"
1,Jumanji,,8844,"[Adventure, Fantasy, Family]"
2,Grumpier Old Men,Grumpy Old Men Collection,15602,"[Romance, Comedy]"
3,Waiting to Exhale,,31357,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II,Father of the Bride Collection,11862,[Comedy]


In [99]:
sample_size = 15000
sample_df = movies_metadata.sample(n=sample_size, replace=False, random_state=490)

sample_df = sample_df.reset_index()
sample_df = sample_df.drop('index', axis=1)

In [100]:
sample_df['title'] = sample_df['title'].str.lower()
sample_df.head()

,title,movieId,genres
0,another happy day,60422,"[Comedy, Drama]"
1,phantoms,9827,"[Horror, Science Fiction, Thriller]"
2,beethoven's christmas adventure,78080,"[Comedy, Family]"
3,role models,15373,[Comedy]
4,"dear guest, when will you leave?",34144,"[Comedy, Foreign]"


In [101]:
sample_df['genres'] = sample_df['genres'].apply(lambda genres: [genre.lower() for genre in genres])
sample_df['genres'] = sample_df['genres'].apply(lambda x: ', '.join(x))
sample_df.head()

,title,movieId,genres
0,another happy day,60422,"comedy, drama"
1,phantoms,9827,"horror, science fiction, thriller"
2,beethoven's christmas adventure,78080,"comedy, family"
3,role models,15373,comedy
4,"dear guest, when will you leave?",34144,"comedy, foreign"


In [105]:
sample_df2 = sample_df
sample_df2.head()

,title,movieId,genres
0,another happy day,60422,"comedy, drama"
1,phantoms,9827,"horror, science fiction, thriller"
2,beethoven's christmas adventure,78080,"comedy, family"
3,role models,15373,comedy
4,"dear guest, when will you leave?",34144,"comedy, foreign"


In [106]:
sample_df2 = sample_df2.drop('movieId', axis=1)

In [107]:
sample_df2['data'] = sample_df2[sample_df2.columns[0:]].apply(lambda x: ' '.join(x.dropna().astype(str).str.replace(',', '')), axis=1)


In [108]:
sample_df2.head(10)

,title,genres,data
0,another happy day,"comedy, drama",another happy day comedy drama
1,phantoms,"horror, science fiction, thriller",phantoms horror science fiction thriller
2,beethoven's christmas adventure,"comedy, family",beethoven's christmas adventure comedy family
3,role models,comedy,role models comedy
4,"dear guest, when will you leave?","comedy, foreign",dear guest when will you leave? comedy foreign
5,race,"drama, action, thriller, crime",race drama action thriller crime
6,rudolph and frosty's christmas in july,"animation, fantasy, family",rudolph and frosty's christmas in july animati...
7,scuola di ladri 2,,scuola di ladri 2
8,straight time,"drama, crime",straight time drama crime
9,chasing freedom,drama,chasing freedom drama


In [109]:
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(sample_df2['data'])

In [111]:
similarities = cosine_similarity(vectorized)

In [113]:
sample_df = pd.DataFrame(similarities, columns=sample_df['title'], index=sample_df['title']).reset_index()

sample_df.head(50)

title,title,another happy day,phantoms,beethoven's christmas adventure,role models,"dear guest, when will you leave?",race,rudolph and frosty's christmas in july,scuola di ladri 2,straight time,...,karachi se lahore,kajaki,wrong turn 6: last resort,the phantom,sweet revenge,attack force z,loco love,the third miracle,santa paws 2: the santa pups,cruel summer
0,another happy day,1.000000,0.0,0.200000,0.258199,0.158114,0.2,0.000000,0.0,0.223607,...,0.200000,0.2,0.0,0.258199,0.182574,0.2,0.223607,0.223607,0.000000,0.2
1,phantoms,0.000000,1.0,0.000000,0.000000,0.000000,0.2,0.000000,0.0,0.000000,...,0.000000,0.2,0.2,0.000000,0.182574,0.0,0.000000,0.000000,0.000000,0.4
2,beethoven's christmas adventure,0.200000,0.0,1.000000,0.258199,0.158114,0.0,0.298142,0.0,0.000000,...,0.400000,0.2,0.0,0.000000,0.182574,0.0,0.223607,0.000000,0.158114,0.0
3,role models,0.258199,0.0,0.258199,1.000000,0.204124,0.0,0.000000,0.0,0.000000,...,0.258199,0.0,0.0,0.000000,0.000000,0.0,0.288675,0.000000,0.000000,0.0
4,"dear guest, when will you leave?",0.158114,0.0,0.158114,0.204124,1.000000,0.0,0.000000,0.0,0.000000,...,0.158114,0.0,0.0,0.000000,0.000000,0.0,0.176777,0.000000,0.000000,0.0


In [123]:
input_movie = 'another happy day'
recommendations = pd.DataFrame(sample_df.nlargest(11, input_movie)['title'])
recommendations = recommendations[recommendations['title'] != input_movie]
print(recommendations)

                          title
6608   not another happy ending
3037                another you
4117               another face
5706               another year
9217              a happy event
12630                fool's day
12228                    c.o.g.
3628              day for night
3668             a day in court
5502             happy-go-lucky


In [4]:
movie_matrix = movies_metadata
movie_matrix['title'] = movie_matrix['title'].str.lower()
movie_matrix.head()

,title,collection,movieId,genres
0,toy story,Toy Story Collection,862,"[Animation, Comedy, Family]"
1,jumanji,,8844,"[Adventure, Fantasy, Family]"
2,grumpier old men,Grumpy Old Men Collection,15602,"[Romance, Comedy]"
3,waiting to exhale,,31357,"[Comedy, Drama, Romance]"
4,father of the bride part ii,Father of the Bride Collection,11862,[Comedy]


In [5]:
movie_matrix['genres'] = movie_matrix['genres'].apply(lambda genres: [genre.lower() for genre in genres])
movie_matrix['genres'] = movie_matrix['genres'].apply(lambda x: ', '.join(x))

movie_matrix['collection'] = movie_matrix['collection'].apply(lambda x: x.lower())
movie_matrix.head()

,title,collection,movieId,genres
0,toy story,toy story collection,862,"animation, comedy, family"
1,jumanji,,8844,"adventure, fantasy, family"
2,grumpier old men,grumpy old men collection,15602,"romance, comedy"
3,waiting to exhale,,31357,"comedy, drama, romance"
4,father of the bride part ii,father of the bride collection,11862,comedy


In [6]:
movie_matrix2 = movie_matrix

In [7]:
movie_matrix2 = movie_matrix2.drop('movieId', axis=1)

In [8]:
movie_matrix2['data'] = movie_matrix2[movie_matrix2.columns[0:]].apply(lambda x: ' '.join(x.dropna().astype(str).str.replace(',', '')), axis=1)

In [9]:
# vectorizer = CountVectorizer()
vectorizer = TfidfVectorizer()
vectorized = vectorizer.fit_transform(movie_matrix2['data'])

In [10]:
similarities = cosine_similarity(vectorized)

In [11]:
movie_matrix = pd.DataFrame(similarities, columns=movie_matrix['title'], index=movie_matrix['title'])

In [17]:
movie_matrix.head(10)

title,toy story,jumanji,grumpier old men,waiting to exhale,father of the bride part ii,heat,sabrina,tom and huck,sudden death,goldeneye,...,house of horrors,shadow of the blair witch,the burkittsville 7,caged heat 3000,robin hood,subdue,century of birthing,betrayal,satan triumphant,queerama
title,,,,,,,,,,,,,,,,,,,,,
toy story,1.000000,0.144338,0.160128,0.117851,0.129099,0.000000,0.166667,0.109109,0.000000,0.109109,...,0.000000,0.000000,0.000000,0.0,0.000000,0.166667,0.000000,0.000000,0.0,0.0
jumanji,0.144338,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.377964,0.223607,0.188982,...,0.000000,0.000000,0.000000,0.0,0.000000,0.288675,0.000000,0.000000,0.0,0.0
grumpier old men,0.160128,0.000000,1.000000,0.226455,0.124035,0.000000,0.320256,0.000000,0.000000,0.104828,...,0.000000,0.000000,0.000000,0.0,0.124035,0.000000,0.000000,0.000000,0.0,0.0
waiting to exhale,0.117851,0.000000,0.226455,1.000000,0.091287,0.182574,0.471405,0.154303,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.365148,0.235702,0.204124,0.204124,0.0,0.0
father of the bride part ii,0.129099,0.000000,0.124035,0.091287,1.000000,0.000000,0.129099,0.000000,0.000000,0.084515,...,0.182574,0.338062,0.258199,0.0,0.000000,0.000000,0.223607,0.000000,0.0,0.0
heat,0.000000,0.000000,0.000000,0.182574,0.000000,1.000000,0.000000,0.338062,0.400000,0.338062,...,0.182574,0.000000,0.000000,0.2,0.400000,0.258199,0.223607,0.670820,0.0,0.0
sabrina,0.166667,0.000000,0.320256,0.471405,0.129099,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.258199,0.000000,0.000000,0.000000,0.0,0.0
tom and huck,0.109109,0.377964,0.000000,0.154303,0.000000,0.338062,0.000000,1.000000,0.338062,0.285714,...,0.000000,0.000000,0.000000,0.0,0.338062,0.436436,0.188982,0.377964,0.0,0.0
sudden death,0.000000,0.223607,0.000000,0.000000,0.000000,0.400000,0.000000,0.338062,1.000000,0.507093,...,0.182574,0.000000,0.000000,0.0,0.200000,0.000000,0.000000,0.447214,0.0,0.0


In [14]:
movie_matrix.info()

<class 'pandas.DataFrame'>
Index: 45463 entries, toy story to queerama
Columns: 45463 entries, toy story to queerama
dtypes: float64(45463)
memory usage: 15.4 GB


In [18]:
input_movie = 'avengers: age of ultron'
recommendations = movie_matrix[input_movie].nlargest(11)
recommendations = recommendations[recommendations.index != input_movie]
print(recommendations)

title
the avengers                         0.816020
3 avengers                           0.749673
the avengers                         0.718759
ultimate avengers                    0.597517
ultimate avengers 2                  0.597517
the shaolin avengers                 0.568822
avengers grimm                       0.503425
crippled avengers                    0.497632
masked avengers                      0.496211
next avengers: heroes of tomorrow    0.414939
Name: avengers: age of ultron, dtype: float64


In [2]:
df = pd.read_csv("dataset/movies/movies_metadata.csv")

df = df[['title', 'belongs_to_collection', 'id', 'genres']]
df = df.rename(columns={"id": "movieId", "belongs_to_collection": "collection"})

df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce')
df = df.dropna(subset=['movieId'])
df['movieId'] = df['movieId'].astype('int64')

df.head()

/var/folders/7k/s2jvnzk97s987g3tmxp90kdw0000gn/T/ipykernel_59955/1549490411.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dataset/movies/movies_metadata.csv")


,title,collection,movieId,genres
0,Toy Story,"{'id': 10194, 'name': 'Toy Story Collection', ...",862,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,NaN,8844,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",15602,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,NaN,31357,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,"{'id': 96871, 'name': 'Father of the Bride Col...",11862,"[{'id': 35, 'name': 'Comedy'}]"


In [5]:
df['genres'] = df['genres'].apply(lambda x: [item['name'] for item in ast.literal_eval(x)])
df['collection'] = df['collection'].apply(lambda x: ast.literal_eval(x)['name'] if pd.notna(x) and x != '' else '')

In [6]:
df_matrix = df
df_matrix['title'] = df_matrix['title'].str.lower()

df_matrix.head()

,title,collection,movieId,genres
0,toy story,Toy Story Collection,862,"[Animation, Comedy, Family]"
1,jumanji,,8844,"[Adventure, Fantasy, Family]"
2,grumpier old men,Grumpy Old Men Collection,15602,"[Romance, Comedy]"
3,waiting to exhale,,31357,"[Comedy, Drama, Romance]"
4,father of the bride part ii,Father of the Bride Collection,11862,[Comedy]


In [7]:
df_matrix['genres'] = df_matrix['genres'].apply(lambda genres: [genre.lower() for genre in genres])
df_matrix['genres'] = df_matrix['genres'].apply(lambda x: ', '.join(x))

df_matrix['collection'] = df_matrix['collection'].apply(lambda x: x.lower())

df_matrix.head()

,title,collection,movieId,genres
0,toy story,toy story collection,862,"animation, comedy, family"
1,jumanji,,8844,"adventure, fantasy, family"
2,grumpier old men,grumpy old men collection,15602,"romance, comedy"
3,waiting to exhale,,31357,"comedy, drama, romance"
4,father of the bride part ii,father of the bride collection,11862,comedy


In [8]:
df_matrix2 = df_matrix

In [9]:
df_matrix2 = df_matrix2.drop('movieId', axis=1)

In [12]:
df_matrix2.head()

,title,collection,genres
0,toy story,toy story collection,"animation, comedy, family"
1,jumanji,,"adventure, fantasy, family"
2,grumpier old men,grumpy old men collection,"romance, comedy"
3,waiting to exhale,,"comedy, drama, romance"
4,father of the bride part ii,father of the bride collection,comedy


In [10]:
df_matrix2['data'] = df_matrix2[df_matrix2.columns[1:]].apply(lambda x: ' '.join(x.dropna().astype(str).str.replace(',', '')), axis=1)

In [11]:
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df_matrix2['data'])

In [12]:
similarities = cosine_similarity(vectorized)

In [13]:
df_matrix = pd.DataFrame(similarities, columns=df_matrix['title'], index=df_matrix['title'])

In [18]:
df_matrix.head()

title,toy story,jumanji,grumpier old men,waiting to exhale,father of the bride part ii,heat,sabrina,tom and huck,sudden death,goldeneye,...,house of horrors,shadow of the blair witch,the burkittsville 7,caged heat 3000,robin hood,subdue,century of birthing,betrayal,satan triumphant,queerama
title,,,,,,,,,,,,,,,,,,,,,
toy story,1.000000,0.235702,0.333333,0.235702,0.333333,0.000000,0.288675,0.204124,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.000000,0.288675,0.00000,0.000000,0.0,0.0
jumanji,0.235702,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.577350,0.333333,0.235702,...,0.0,0.0,0.0,0.0,0.000000,0.408248,0.00000,0.000000,0.0,0.0
grumpier old men,0.333333,0.000000,1.000000,0.471405,0.333333,0.000000,0.577350,0.000000,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.235702,0.000000,0.00000,0.000000,0.0,0.0
waiting to exhale,0.235702,0.000000,0.471405,1.000000,0.235702,0.288675,0.816497,0.288675,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.666667,0.408248,0.57735,0.333333,0.0,0.0
father of the bride part ii,0.333333,0.000000,0.333333,0.235702,1.000000,0.000000,0.288675,0.000000,0.000000,0.166667,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0


In [23]:
input_movie = 'the godfather'
recommendations = df_matrix[input_movie.lower()].nlargest(11)
recommendations = recommendations[recommendations.index != input_movie.lower()]
print(recommendations)

title
the godfather: part ii     1.000000
the godfather: part iii    0.912871
veronika voss              0.800000
the sting                  0.730297
the skulls                 0.730297
salting the battlefield    0.730297
sweeney!                   0.730297
the two jakes              0.676123
the exterminator           0.676123
enter the ninja            0.676123
Name: the godfather, dtype: float64
